Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('..')

Load the data

In [94]:
from src.dataloader import DataLoader
from src.config import load_config

config = load_config('configs/base.yaml')

dataloader = DataLoader(
    label_path=config['paths']['label'],
    data_path=config['paths']['data'],
    start_date=config['data']['start_date'],
    end_date=config['data']['end_date'],
    download=config['data']['download'],
    load=config['data']['load'],
    normalize=config['data']['normalize'],
    debug=True
)

100%|██████████| 70/70 [01:02<00:00,  1.12it/s]


Create a temporal graph with node features and targets

In [172]:
from src.config import load_config
from src.graph import TemporalGraphNormalize

config = load_config('configs/base.yaml')
data = dataloader.get_graph(k=config['graph']['k'],
                            r=config['graph']['r']
                            )

# Apply normalization transform
normalize_transform = TemporalGraphNormalize(normalize=True, fill_nan='both')
data = normalize_transform(data)

Transform the temporal graph into a graph with static features from (a) the signature, and (b) random features

In [194]:
from src.signature import SignatureFeatures, RandomFeatures
from src.signature import RandomFeatures
from signatory import signature_channels
from src.config import load_config

hyper = load_config('configs/hyper.yaml')

# Configure the signature transform as needed
sig_transform = SignatureFeatures(
    sig_depth=hyper['sig']['depth'],         # or your desired depth
    normalize=hyper['sig']['normalize'],      # whether to normalize the signature features
    log_signature=hyper['sig']['log_signature'], # use log-signature or not
    time_augment=hyper['sig']['time_augment'],  # add time as a feature or not
    lead_lag=hyper['sig']['lead_lag']      # use lead-lag augmentation or not
)

# Apply the transform to your temporal graph data
static_graph = sig_transform(data)

In [200]:
static_graph.num_node_features

1554

Train a GCN classifier to predict labels from signature features

In [204]:
import torch
from signatory import signature_channels
from src.graph import NodeSplitMask
from src.model import ClassifierGCN


hyper = load_config('configs/hyper.yaml')

split_transform = NodeSplitMask(train_ratio=hyper['split']['train_ratio'],
                                val_ratio=hyper['split']['val_ratio'],
                                test_ratio=hyper['split']['test_ratio'],
                                seed=hyper['split']['seed'])
static_graph = split_transform(static_graph)


model = ClassifierGCN(node_features=static_graph.num_node_features,
                      hidden_features=hyper['hidden_features'],
                      num_classes=hyper['num_classes'])

optimizer = torch.optim.Adam(model.parameters(),
                             lr=hyper['lr'],
                             weight_decay=hyper['weight_decay'])
criterion = torch.nn.CrossEntropyLoss()

# Training loop
epochs = hyper
for epoch in range(int(hyper['num_epochs'])):
    model.train()
    optimizer.zero_grad()
    out = model(static_graph.x, static_graph.edge_index, static_graph.edge_attr)
    # Only use labeled nodes for training
    loss = criterion(out[static_graph.train_mask], static_graph.y[static_graph.train_mask].long())
    loss.backward()
    optimizer.step()

    # Validation
    model.eval()
    with torch.no_grad():
        pred = out.argmax(dim=1)
        train_acc = (pred[static_graph.train_mask] == static_graph.y[static_graph.train_mask]).float().mean().item()
        val_acc = (pred[static_graph.val_mask] == static_graph.y[static_graph.val_mask]).float().mean().item() if static_graph.val_mask.sum() > 0 else float('nan')
    if epoch % 10 == 0:
        print(f"Epoch {epoch:03d} | Loss: {loss.item():.4f} | Train Acc: {train_acc:.4f}")

# Test evaluation
model.eval()
with torch.no_grad():
    out = model(static_graph.x, static_graph.edge_index, static_graph.edge_attr)
    pred = out.argmax(dim=1)
    test_acc = (pred[static_graph.test_mask] == static_graph.y[static_graph.test_mask]).float().mean().item() if static_graph.test_mask.sum() > 0 else float('nan')
print(f"Test Accuracy: {test_acc:.4f}")

Epoch 000 | Loss: 0.5785 | Train Acc: 0.8000
Epoch 010 | Loss: 0.4601 | Train Acc: 0.9091
Epoch 020 | Loss: 0.3958 | Train Acc: 0.8727
Epoch 030 | Loss: 0.3528 | Train Acc: 0.8727
Epoch 040 | Loss: 0.3216 | Train Acc: 0.8545
Epoch 050 | Loss: 0.2975 | Train Acc: 0.8909
Epoch 060 | Loss: 0.2783 | Train Acc: 0.9273
Epoch 070 | Loss: 0.2623 | Train Acc: 0.9273
Epoch 080 | Loss: 0.2486 | Train Acc: 0.9273
Epoch 090 | Loss: 0.2364 | Train Acc: 0.9273
Epoch 100 | Loss: 0.2252 | Train Acc: 0.9273
Epoch 110 | Loss: 0.2147 | Train Acc: 0.9273
Epoch 120 | Loss: 0.2047 | Train Acc: 0.9273
Epoch 130 | Loss: 0.1952 | Train Acc: 0.9273
Epoch 140 | Loss: 0.1859 | Train Acc: 0.9273
Epoch 150 | Loss: 0.1769 | Train Acc: 0.9273
Epoch 160 | Loss: 0.1685 | Train Acc: 0.9273
Epoch 170 | Loss: 0.1607 | Train Acc: 0.9273
Epoch 180 | Loss: 0.1532 | Train Acc: 0.9273
Epoch 190 | Loss: 0.1462 | Train Acc: 0.9273
Epoch 200 | Loss: 0.1395 | Train Acc: 0.9455
Epoch 210 | Loss: 0.1331 | Train Acc: 0.9455
Epoch 220 

In [205]:
pred

tensor([0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0,
        0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [207]:
data[0].y

tensor([0., 0., 1., 0., 0., 0., 0., 0., 1., 0., 1., 0., 1., 0., 0., 0., 0., 1.,
        0., 0., 0., 1., 0., 0., 0., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 1., 0., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       dtype=torch.float64)